In [ ]:
import io
from io import BytesIO

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from PIL import Image
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import MolDrawOptions

from src.core.utils import select_diverse_subset_butina

def get_diverse_mols(smiles_list, num_examples=5):
    smiles_ids = select_diverse_subset_butina(
        [Chem.MolFromSmiles(smiles) for smiles in smiles_list],
        num_to_select=num_examples, similarity_cutoff=0.3
    )
    mols = [Chem.MolFromSmiles(smiles_list[i]) for i in smiles_ids]
    return mols


def visualize_molecules(mols, per_row=5, subim_size=(200,200)):
    options = MolDrawOptions()
    options.padding = 0.01
    img = Draw.MolsToGridImage(mols, molsPerRow=per_row, subImgSize=subim_size, drawOptions=options)

    png_data = img.data
    img_arr = mpimg.imread(io.BytesIO(png_data), format='png')

    mask = np.any(img_arr[:, :, :3] != 1, axis=2)

    # Find the rows and columns that contain content
    rows_with_content = np.any(mask, axis=1)
    cols_with_content = np.any(mask, axis=0)

    # Find the min/max indices of the content
    if np.any(rows_with_content):
        ymin, ymax = np.where(rows_with_content)[0][[0, -1]]
        xmin, xmax = np.where(cols_with_content)[0][[0, -1]]

        # Add a tiny 1-pixel border for aesthetics
        ymin = max(0, ymin - 1)
        ymax = min(img_arr.shape[0], ymax + 2)
        xmin = max(0, xmin - 1)
        xmax = min(img_arr.shape[1], xmax + 2)

        # 4. Slice the array to the content box
        cropped_arr = img_arr[ymin:ymax, xmin:xmax]
        return cropped_arr
    else:
        # Return the original image if it's all white (or empty)
        return img_arr

# QM9 data

In [ ]:
datasets_paths = {
    'Linear': 'qm9_simple_linear6.csv',
    'Piecewise Linear': 'qm9_piecewise_linear_6.csv',
    'Polynomial': 'qm9_nonlinear_6.csv',
}

In [ ]:
import pandas as pd

datasets = {}
for n, d in datasets_paths.items():
    df = pd.read_csv(f'../data/synthetic_data/{d}')
    print(n, df.shape)
    df = df.dropna().reset_index(drop=True)
    datasets[n] = df

smiles_list = datasets['Linear']['smiles'].tolist()
smiles_list

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(12, 7))

gs = gridspec.GridSpec(2, 3, figure=fig, height_ratios=[1, 1], width_ratios=[1,1,1], hspace=0.2, wspace=0.3)

#exemplary molecules
ax1 = fig.add_subplot(gs[0, :])
ax1.set_title('Exemplary Molecules', fontsize=14, pad=20, weight='bold')

# Generate and display the image
mols = get_diverse_mols(smiles_list, num_examples=6)
img = visualize_molecules(mols, per_row=6)
ax1.imshow(img)
ax1.axis('off')

# Target distributions
fig.suptitle('Target Distributions', y=0.55, fontsize=14, weight='bold')
for i, (name, df) in enumerate(datasets.items()):
    ax = fig.add_subplot(gs[1, i])
    sns.histplot(data=df, x='target', bins=50, kde=True, ax=ax)
    ax.set_title(f'{name}', fontsize=12)
    ax.set_xlabel('Target')
    ax.set_ylabel('Frequency')

plt.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig('qm9_datasets_overview.pdf', dpi=300, bbox_inches='tight')
plt.show()

# COF data

In [ ]:
cof_path = '../data/cof_data/data_batteries_ecfp_descriptor.csv'

df_cof = pd.read_csv(cof_path)
smiles_list = df_cof['smiles'].tolist()
smiles_list

In [ ]:
fig = plt.figure(figsize=(60, 30))

# Use the 4x2 grid for proportionate layout
gs = gridspec.GridSpec(4, 2, figure=fig, hspace=0.05, wspace=0.1,
                       width_ratios=[2, 1.5],
                       height_ratios=[0.5, 1, 1, 0.5])

ax1 = fig.add_subplot(gs[0:2, 0])

mols = get_diverse_mols(smiles_list, num_examples=4)
img1 = visualize_molecules(mols[:2], per_row=2, subim_size=(400,400))
ax1.imshow(img1)
ax1.axis('off')

ax2 = fig.add_subplot(gs[2:4, 0])
img2 = visualize_molecules(mols[2:], per_row=2, subim_size=(400,400))
ax2.imshow(img2)
ax2.axis('off')

ax_hist = fig.add_subplot(gs[1:3, 1])

sns.histplot(data=df_cof, x='capacity_max', bins=50, kde=True, ax=ax_hist)
ax_hist.set_xlabel('Capacitance (F/g)', fontsize=55)
ax_hist.set_ylabel('Frequency', fontsize=55)
ax_hist.tick_params(axis='both', which='major', labelsize=50)


plt.tight_layout(rect=[0, 0, 1, 0.93])


pos_mols = ax1.get_position()
pos_hist = ax_hist.get_position()
x_center_mols = pos_mols.x0 + pos_mols.width / 2
x_center_hist = pos_hist.x0 + pos_hist.width / 2

title_y_position = 0.87 # A single vertical position for both titles

fig.text(x_center_mols, title_y_position, 'Exemplary Molecules',
         ha='center', va='top', fontsize=70, weight='bold')

fig.text(x_center_hist, title_y_position, 'Target Distribution',
         ha='center', va='top', fontsize=70, weight='bold')

fig.savefig('cof_dataset_overview.pdf', dpi=300, bbox_inches='tight', pad_inches=0)
plt.show()

# hERG data

In [ ]:
herg_path = '../data/herg_data/data_herg_ecfp.csv'

df_herg = pd.read_csv(herg_path)
smiles_list = df_herg['smiles'].tolist()
smiles_list

In [ ]:
fig = plt.figure(figsize=(60, 30))

# Use the 4x2 grid for proportionate layout
gs = gridspec.GridSpec(4, 2, figure=fig, hspace=0.05, wspace=0.1,
                       width_ratios=[2, 1.5],
                       height_ratios=[0.5, 1, 1, 0.5])

ax1 = fig.add_subplot(gs[0:2, 0])

mols = get_diverse_mols(smiles_list, num_examples=4)
img1 = visualize_molecules(mols[:2], per_row=2, subim_size=(400,400))
ax1.imshow(img1)
ax1.axis('off')

ax2 = fig.add_subplot(gs[2:4, 0])
img2 = visualize_molecules(mols[2:], per_row=2, subim_size=(400,400))
ax2.imshow(img2)
ax2.axis('off')

ax_hist = fig.add_subplot(gs[1:3, 1])

sns.histplot(data=df_herg, x='pic50', bins=50, kde=True, ax=ax_hist)
ax_hist.set_xlabel('pIC50', fontsize=55)
ax_hist.set_ylabel('Frequency', fontsize=55)
ax_hist.tick_params(axis='both', which='major', labelsize=50)


plt.tight_layout(rect=[0, 0, 1, 0.93])


pos_mols = ax1.get_position()
pos_hist = ax_hist.get_position()
x_center_mols = pos_mols.x0 + pos_mols.width / 2
x_center_hist = pos_hist.x0 + pos_hist.width / 2

title_y_position = 0.87 # A single vertical position for both titles

fig.text(x_center_mols, title_y_position, 'Exemplary Molecules',
         ha='center', va='top', fontsize=70, weight='bold')

fig.text(x_center_hist, title_y_position, 'Target Distribution',
         ha='center', va='top', fontsize=70, weight='bold')

fig.savefig('herg_dataset_overview.pdf', dpi=300, bbox_inches='tight', pad_inches=0)
plt.show()